# Checkpoint / History Audit — Step 1 of reproducibility fix

Goal: figure out **which trained checkpoint is the one whose numbers ended up in the hEART paper**,
before regenerating or editing anything.

This notebook:
1. Lists modified timestamps for every checkpoint, history file, and results CSV, so you can see which
   artifacts were produced together (same run) vs. which look orphaned/stale.
2. Loads each `*_history.pkl` and prints epoch counts / final loss values.
3. Loads each `*_best.pt` checkpoint and prints any stored metadata (epoch, val_loss, seed, timestamp —
   whatever `train.py` actually saved).
4. Gives you a place to record the decision once you've made it.

It does **not** retrain or re-evaluate anything by default — this is read-only reconnaissance.

In [1]:
import os, glob, pickle
from datetime import datetime
import torch
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/chicago-stt-cta-replicability"  # update if you renamed/moved the folder
OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

MODEL_NAMES = ["historical_average", "speed_only", "demand_only", "independent_dual", "joint"]

assert os.path.isdir(CKPT_DIR), f"Checkpoint dir not found: {CKPT_DIR} — check PROJECT_DIR above."

Mounted at /content/drive


## 1. Timeline of every relevant artifact

If checkpoints, history files, and the results CSVs all cluster around the same date/time, they're
probably from the same run. A checkpoint that's much older or newer than the CSVs it supposedly
produced is the first thing to be suspicious of.

In [2]:
def file_info(path):
    if not os.path.exists(path):
        return {"path": path, "size_kb": None, "modified": None, "status": "MISSING"}
    stat = os.stat(path)
    return {
        "path": os.path.relpath(path, PROJECT_DIR),
        "size_kb": round(stat.st_size / 1024, 1),
        "modified": datetime.fromtimestamp(stat.st_mtime).isoformat(sep=" "),
        "status": "ok",
    }

artifact_paths = []
artifact_paths += sorted(glob.glob(os.path.join(CKPT_DIR, "*")))
artifact_paths += [
    os.path.join(OUTPUT_DIR, "evaluation_results.csv"),
    os.path.join(OUTPUT_DIR, "ha_vs_joint_results.csv"),
    os.path.join(PROJECT_DIR, "paper.tex"),
    os.path.join(PROJECT_DIR, "config.py"),
]

timeline = pd.DataFrame([file_info(p) for p in artifact_paths])
timeline = timeline.sort_values("modified", na_position="last").reset_index(drop=True)
timeline

,path,size_kb,modified,status
0,config.py,6.9,2026-07-26 13:13:14,ok
1,output/checkpoints/historical_average_best.pt,71.2,2026-07-26 13:14:07,ok
2,output/checkpoints/historical_average_history.pkl,0.0,2026-07-26 13:14:10,ok
3,output/checkpoints/speed_only_best.pt,2816.3,2026-07-26 13:14:14,ok
4,output/checkpoints/speed_only_history.pkl,2.5,2026-07-26 13:14:17,ok
5,output/checkpoints/demand_only_best.pt,2666.2,2026-07-26 13:14:21,ok
6,output/checkpoints/demand_only_history.pkl,2.5,2026-07-26 13:14:25,ok
7,output/checkpoints/independent_dual_best.pt,5496.2,2026-07-26 13:14:29,ok
8,output/checkpoints/independent_dual_history.pkl,3.2,2026-07-26 13:14:32,ok
9,output/checkpoints/joint_best.pt,5556.0,2026-07-26 13:14:36,ok


## 2. Training history per model

Loads each `*_history.pkl`. Structure depends on what `train.py` actually pickled — this handles the
common case (dict of lists like `train_loss`, `val_loss` per epoch) and falls back to printing raw type
info if the structure is different, so nothing silently fails.

In [3]:
history_rows = []
raw_histories = {}

for name in MODEL_NAMES:
    hist_path = os.path.join(CKPT_DIR, f"{name}_history.pkl")
    if not os.path.exists(hist_path):
        history_rows.append({"model": name, "status": "MISSING"})
        continue
    with open(hist_path, "rb") as f:
        hist = pickle.load(f)
    raw_histories[name] = hist

    row = {"model": name, "status": "ok"}
    if isinstance(hist, dict):
        for k, v in hist.items():
            if isinstance(v, (list, tuple)):
                row[f"{k}_epochs"] = len(v)
                row[f"{k}_final"] = round(v[-1], 4) if len(v) and isinstance(v[-1], (int, float)) else v[-1] if len(v) else None
            else:
                row[k] = v
    else:
        row["raw_type"] = str(type(hist))
    history_rows.append(row)

pd.DataFrame(history_rows)

,model,status,train_loss_epochs,train_loss_final,val_loss_epochs,val_loss_final,train_speed_epochs,train_speed_final,val_speed_epochs,val_speed_final,train_demand_epochs,train_demand_final,val_demand_epochs,val_demand_final,lr_epochs,lr_final
0,historical_average,ok,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,speed_only,ok,39,0.0121,39,0.0115,39.0,0.0121,39.0,0.0115,39.0,0.0000,39.0,0.0000,39.0,0.0
2,demand_only,ok,38,0.0728,38,0.0571,38.0,0.0000,38.0,0.0000,38.0,0.0728,38.0,0.0571,38.0,0.0
3,independent_dual,ok,50,-0.6266,50,-0.6454,50.0,0.0115,50.0,0.0114,50.0,0.0711,50.0,0.0540,50.0,0.0
4,joint,ok,166,-2.1484,166,-2.0200,166.0,0.0101,166.0,0.0121,166.0,0.0436,166.0,0.0551,166.0,0.0


## 3. Checkpoint metadata per model

Loads each `*_best.pt` with `torch.load` and prints any non-tensor top-level keys (epoch, val_loss,
seed, timestamp — whatever was actually saved alongside the state dict). If `train.py` only saved a
raw `state_dict`, this will just show `top_level_keys` as the model's own layer names, which tells you
there's no embedded metadata to lean on — you'll need to rely on file timestamps and the history pkl
instead.

In [9]:
ckpt_rows = []

for name in MODEL_NAMES:
    ckpt_path = os.path.join(CKPT_DIR, f"{name}_best.pt")
    if not os.path.exists(ckpt_path):
        ckpt_rows.append({"model": name, "status": "MISSING"})
        continue
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    row = {"model": name, "status": "ok"}
    if isinstance(ckpt, dict):
        row["top_level_keys"] = list(ckpt.keys())[:10]
        for key in ["epoch", "val_loss", "best_val_loss", "train_loss", "timestamp", "seed", "date", "config"]:
            if key in ckpt:
                row[key] = ckpt[key]
    else:
        row["raw_type"] = str(type(ckpt))
    ckpt_rows.append(row)

pd.DataFrame(ckpt_rows)

RuntimeError: Invalid magic number; corrupt file?

## 4. How to read the results above

- **Timeline (section 1):** if `evaluation_results.csv` is newer than the checkpoints, it was
  (re)computed from *some* checkpoint — check whether all five `*_best.pt` files share that same
  approximate modified time. If one model's checkpoint is noticeably older/newer than the rest, that
  model's row in the CSV may not correspond to the same run as the others.
- **History (section 2):** compare `val_loss_final` (or equivalent) across models — does the `joint`
  model's final val loss look consistent with it being a fully-converged, early-stopped run (per
  `config.py`'s `PATIENCE = 10`), or does it look like a partial/interrupted run?
- **Checkpoint metadata (section 3):** if `epoch` or `timestamp` is present, cross-check it against the
  history file's epoch count for the same model — they should match (checkpoint should be from the
  best epoch found during that same training history).

If none of the checkpoints store explicit timestamps/epochs, the file-system `modified` time from
section 1 is your best signal — checkpoints and the CSV that reports their performance should cluster
tightly in time.

## 5. Record the decision

Once you've looked at sections 1–3, fill this in and keep it — this becomes the canonical-run record
referenced in `KNOWN_ISSUES.md` before moving to Step 2 (regenerate results from this checkpoint).

In [ ]:
CANONICAL_RUN_DATE = None   # e.g. "2026-02-12" — fill in once decided
CANONICAL_NOTES = ""        # e.g. "All 5 *_best.pt checkpoints modified within 40min of each other on
                             #        2026-02-12; evaluation_results.csv regenerated 2026-04-15 from these
                             #        same files (unchanged mtimes on checkpoints since 02-12)."

print("Canonical run date:", CANONICAL_RUN_DATE)
print("Notes:", CANONICAL_NOTES)

# Next: append this decision to KNOWN_ISSUES.md, then proceed to Step 2
# (re-run evaluate.py / evaluate_ha_vs_joint.py against this checkpoint set).

Canonical run date: None
Notes: 
